<a href="https://colab.research.google.com/github/M-Santhoshkumar-2005/ITA0616-MACHINE-LEARNING-SLOT-A-/blob/main/Experiment_15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd


dataset = [
    [5.1, 3.5, 1.4, 0.2, "Iris-setosa"],
    [4.9, 3.0, 1.4, 0.2, "Iris-setosa"],
    [4.7, 3.2, 1.3, 0.2, "Iris-setosa"],
    [4.6, 3.1, 1.5, 0.2, "Iris-setosa"],
    [5.0, 3.6, 1.4, 0.2, "Iris-setosa"],
    [5.4, 3.9, 1.7, 0.4, "Iris-setosa"],
    [7.0, 3.2, 4.7, 1.4, "Iris-versicolor"],
    [6.4, 3.2, 4.5, 1.5, "Iris-versicolor"],
    [6.9, 3.1, 4.9, 1.5, "Iris-versicolor"],
    [5.5, 2.3, 4.0, 1.3, "Iris-versicolor"],
    [6.5, 2.8, 4.6, 1.5, "Iris-versicolor"],
    [5.7, 2.8, 4.5, 1.3, "Iris-versicolor"],
    [6.3, 3.3, 6.0, 2.5, "Iris-virginica"],
    [5.8, 2.7, 5.1, 1.9, "Iris-virginica"],
    [7.1, 3.0, 5.9, 2.1, "Iris-virginica"],
    [6.3, 2.9, 5.6, 1.8, "Iris-virginica"],
    [6.5, 3.0, 5.8, 2.2, "Iris-virginica"],
    [7.6, 3.0, 6.6, 2.1, "Iris-virginica"],
]

df = pd.DataFrame(
    dataset,
    columns=["SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm", "Species"],
)

feature_cols = ["SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm"]
test_indices = [4, 5, 10, 11, 16, 17]
train_mask = np.ones(len(df), dtype=bool)
train_mask[test_indices] = False

train_df = df[train_mask].reset_index(drop=True)
test_df = df.iloc[test_indices].reset_index(drop=True)

classes = sorted(train_df["Species"].unique())
priors = {}
stats = {}

for c in classes:
    subset = train_df[train_df["Species"] == c][feature_cols].values.astype(float)
    priors[c] = len(subset) / len(train_df)
    mean_c = np.mean(subset, axis=0)
    var_c = np.var(subset, axis=0) + 1e-6
    stats[c] = (mean_c, var_c)


def gaussian_log_likelihood(x, mean, var):
    return -0.5 * np.sum(np.log(2.0 * np.pi * var) + ((x - mean) ** 2) / var)


def predict_one(x):
    log_posteriors = {}
    for c in classes:
        mean_c, var_c = stats[c]
        log_posteriors[c] = np.log(priors[c]) + gaussian_log_likelihood(x, mean_c, var_c)
    best_class = max(log_posteriors, key=log_posteriors.get)
    max_lp = max(log_posteriors.values())
    exp_scores = {c: np.exp(lp - max_lp) for c, lp in log_posteriors.items()}
    total = sum(exp_scores.values())
    probs = {c: val / total for c, val in exp_scores.items()}
    return best_class, probs


X_test = test_df[feature_cols].values.astype(float)
y_test = test_df["Species"].values

preds = []
probs_list = []
for row in X_test:
    p_cls, p_map = predict_one(row)
    preds.append(p_cls)
    probs_list.append(p_map)

preds = np.array(preds)
accuracy = np.mean(preds == y_test) * 100

print(f"Test Accuracy: {accuracy:.2f}%\n")
for idx, row, actual, pred, p_map in zip(test_indices, X_test, y_test, preds, probs_list):
    rounded_probs = {k: round(v, 4) for k, v in p_map.items()}
    print(
        f"Row {idx + 1:2d} | SL={row[0]:.1f}, SW={row[1]:.1f}, PL={row[2]:.1f}, PW={row[3]:.1f} | "
        f"Actual={actual:15s} | Pred={pred:15s} | Probs={rounded_probs}"
    )

new_samples = np.array([
    [5.0, 3.4, 1.5, 0.2],
    [6.1, 2.9, 4.7, 1.4],
    [6.7, 3.1, 5.6, 2.4],
])

print("\nNew Sample Predictions:")
for i, sample in enumerate(new_samples, start=1):
    p_cls, p_map = predict_one(sample)
    rounded_probs = {k: round(v, 4) for k, v in p_map.items()}
    print(
        f"New {i} | SL={sample[0]:.1f}, SW={sample[1]:.1f}, PL={sample[2]:.1f}, PW={sample[3]:.1f} | "
        f"Pred={p_cls:15s} | Probs={rounded_probs}"
    )

Test Accuracy: 83.33%

Row  5 | SL=5.0, SW=3.6, PL=1.4, PW=0.2 | Actual=Iris-setosa     | Pred=Iris-setosa     | Probs={'Iris-setosa': np.float64(1.0), 'Iris-versicolor': np.float64(0.0), 'Iris-virginica': np.float64(0.0)}
Row  6 | SL=5.4, SW=3.9, PL=1.7, PW=0.4 | Actual=Iris-setosa     | Pred=Iris-virginica  | Probs={'Iris-setosa': np.float64(0.0), 'Iris-versicolor': np.float64(0.0), 'Iris-virginica': np.float64(1.0)}
Row 11 | SL=6.5, SW=2.8, PL=4.6, PW=1.5 | Actual=Iris-versicolor | Pred=Iris-versicolor | Probs={'Iris-setosa': np.float64(0.0), 'Iris-versicolor': np.float64(0.9991), 'Iris-virginica': np.float64(0.0009)}
Row 12 | SL=5.7, SW=2.8, PL=4.5, PW=1.3 | Actual=Iris-versicolor | Pred=Iris-versicolor | Probs={'Iris-setosa': np.float64(0.0), 'Iris-versicolor': np.float64(0.9999), 'Iris-virginica': np.float64(0.0001)}
Row 17 | SL=6.5, SW=3.0, PL=5.8, PW=2.2 | Actual=Iris-virginica  | Pred=Iris-virginica  | Probs={'Iris-setosa': np.float64(0.0), 'Iris-versicolor': np.float64(0.0), 